In [22]:
import wrds
import pandas as pd
import numpy as np


In [23]:
db= wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Done


In [24]:

# ── Step 1: Pull company-level data from comp.funda (ALL years, since 1950) ──
# Size variables: at, revt, emp, seq, ceq, mkvalt, sale, ni, csho

query_funda = """
    SELECT gvkey, conm, fyear, datadate,
           naicsh, sich,
           at, revt, emp, seq, ceq,
           mkvalt, sale, ni, csho,
           popsrc
    FROM comp.funda
    WHERE indfmt = 'INDL'
      AND datafmt = 'STD'
      AND consol = 'C'
"""

df_funda = db.raw_sql(query_funda)
print(f"comp.funda shape: {df_funda.shape}")
print(f"Unique companies: {df_funda['gvkey'].nunique():,}")
print(f"Year range: {df_funda['fyear'].min():.0f} – {df_funda['fyear'].max():.0f}")
print(f"\nBreakdown by popsrc:")
for code, label in [('D','Domestic public'), ('O','Other/Private'), ('I','International')]:
    n = df_funda[df_funda['popsrc']==code]['gvkey'].nunique()
    print(f"  {code} ({label}): {n:,} companies")
df_funda.head()


comp.funda shape: (595176, 16)
Unique companies: 46,678
Year range: 1950 – 2025

Breakdown by popsrc:
  D (Domestic public): 46,678 companies
  O (Other/Private): 0 companies
  I (International): 0 companies
  D (Domestic public): 46,678 companies
  O (Other/Private): 0 companies
  I (International): 0 companies


,gvkey,conm,fyear,datadate,naicsh,sich,at,revt,emp,seq,ceq,mkvalt,sale,ni,csho,popsrc
0,001000,A & E PLASTIK PAK INC,1961,1961-12-31,<NA>,<NA>,<NA>,0.9,<NA>,<NA>,<NA>,<NA>,0.9,<NA>,0.152,D
1,001000,A & E PLASTIK PAK INC,1962,1962-12-31,<NA>,<NA>,<NA>,1.6,<NA>,<NA>,0.552,<NA>,1.6,<NA>,0.181,D
2,001000,A & E PLASTIK PAK INC,1963,1963-12-31,<NA>,<NA>,<NA>,1.457,<NA>,0.553,0.553,<NA>,1.457,0.003,0.186,D
3,001000,A & E PLASTIK PAK INC,1964,1964-12-31,<NA>,<NA>,1.416,2.032,<NA>,0.607,0.607,<NA>,2.032,0.052,0.196,D
4,001000,A & E PLASTIK PAK INC,1965,1965-12-31,<NA>,<NA>,2.31,1.688,<NA>,0.491,0.491,<NA>,1.688,-0.197,0.206,D


In [25]:

# ── Step 2: Pull segment-level NAICS + sales from comp.seg_annfund ──
# This gives us ALL NAICS codes per company-year with sales per segment

seg_query = """
    SELECT f.gvkey, f.stype, f.sid, f.datadate,
           f.sales AS seg_sales, f.revts AS seg_revts,
           n.naicss, n.sics
    FROM comp.seg_annfund f
    LEFT JOIN comp.seg_naics n
        ON f.gvkey = n.gvkey
        AND f.stype = n.stype
        AND f.sid = n.sid
        AND f.datadate = n.datadate
    WHERE f.stype = 'BUSSEG'
"""

df_seg = db.raw_sql(seg_query)
print(f"Segment data shape: {df_seg.shape}")
print(f"Unique companies: {df_seg['gvkey'].nunique():,}")
print(f"Year range: {df_seg['datadate'].min()} – {df_seg['datadate'].max()}")
print(f"Unique NAICS codes: {df_seg['naicss'].nunique():,}")
df_seg.head()


Segment data shape: (279395, 8)
Unique companies: 7,961
Year range: 2017-06-30 – 2025-12-31
Unique NAICS codes: 1,198


,gvkey,stype,sid,datadate,seg_sales,seg_revts,naicss,sics
0,001004,BUSSEG,11,2018-05-31,0.0,0.0,<NA>,<NA>
1,001004,BUSSEG,11,2019-05-31,0.0,0.0,<NA>,<NA>
2,001004,BUSSEG,11,2019-05-31,0.0,0.0,<NA>,<NA>
3,001004,BUSSEG,11,2020-05-31,0.0,0.0,<NA>,<NA>
4,001004,BUSSEG,11,2020-05-31,0.0,0.0,<NA>,<NA>


In [26]:

# ── Step 3: Aggregate segment NAICS & sales per company-year ──
# For each company-year, create:
#   - all_naics: comma-separated list of all NAICS codes
#   - naics_sales: dict-like string mapping each NAICS → its segment sales

# Aggregate: all unique NAICS codes per company-year
naics_list = (df_seg.groupby(['gvkey', 'datadate'])['naicss']
    .apply(lambda x: ', '.join(sorted(x.dropna().unique())))
    .reset_index()
    .rename(columns={'naicss': 'all_naics'})
)
naics_list['naics_count'] = naics_list['all_naics'].apply(
    lambda x: len(x.split(', ')) if x else 0
)

# Aggregate: sales by NAICS per company-year
naics_sales = (df_seg[df_seg['naicss'].notna()]
    .groupby(['gvkey', 'datadate', 'naicss'])['seg_sales']
    .sum()
    .reset_index()
)
naics_sales_agg = (naics_sales
    .groupby(['gvkey', 'datadate'])
    .apply(lambda g: '; '.join(f"{row['naicss']}:{row['seg_sales']:.1f}" 
                                 if pd.notna(row['seg_sales']) 
                                 else f"{row['naicss']}:NA"
                                 for _, row in g.iterrows()))
    .reset_index()
    .rename(columns={0: 'naics_sales'})
)

# Merge the two aggregations
seg_agg = naics_list.merge(naics_sales_agg, on=['gvkey', 'datadate'], how='left')
print(f"Segment aggregation shape: {seg_agg.shape}")
print(f"Sample:")
seg_agg[seg_agg['naics_count'] > 1].head(5)


Segment aggregation shape: (49233, 5)
Sample:


/var/folders/bb/rsmxjk4n2k7_166l026d8h400000gn/T/ipykernel_7275/3757318598.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: '; '.join(f"{row['naicss']}:{row['seg_sales']:.1f}"


,gvkey,datadate,all_naics,naics_count,naics_sales
2,001004,2020-05-31,"336413, 423860, 481211, 488190",4,336413:323.4; 423860:5892.6; 481211:323.4; 488...
3,001004,2021-05-31,"336413, 423860, 481211, 488190",4,336413:295.8; 423860:4661.1; 481211:295.8; 488...
4,001004,2022-05-31,"336413, 423860, 481211, 488190",4,336413:222.6; 423860:3491.6; 481211:222.6; 488...
5,001004,2023-05-31,"336413, 423860, 481211, 488190",4,336413:275.4; 423860:1898.7; 481211:275.4; 488...
6,001004,2024-05-31,"336413, 423860, 488190, 513210",4,336413:139.8; 423860:3222.4; 488190:1280.2; 51...


In [27]:

# ── Step 4: Merge everything into one final database ──
# Company-level (funda) + all segment NAICS codes + sales per NAICS
# LEFT join keeps ALL funda rows; segment NAICS only available from ~2017

df_final = df_funda.merge(seg_agg, on=['gvkey', 'datadate'], how='left')

# Reorder columns
df_final = df_final[['gvkey', 'conm', 'fyear', 'datadate', 'popsrc',
                      'naicsh', 'sich',
                      'all_naics', 'naics_count', 'naics_sales',
                      'at', 'revt', 'emp', 'seq', 'ceq',
                      'mkvalt', 'sale', 'ni', 'csho']]

print(f"═══ FINAL DATABASE ═══")
print(f"Shape: {df_final.shape}")
print(f"Unique companies: {df_final['gvkey'].nunique():,}")
print(f"Year range: {df_final['fyear'].min():.0f} – {df_final['fyear'].max():.0f}")
print(f"\nIndustry code coverage:")
print(f"  naicsh (primary NAICS):  {df_final['naicsh'].notna().sum():,} obs ({df_final['naicsh'].notna().mean()*100:.1f}%)")
print(f"  sich (primary SIC):      {df_final['sich'].notna().sum():,} obs ({df_final['sich'].notna().mean()*100:.1f}%)")
print(f"  all_naics (segment):     {df_final['all_naics'].notna().sum():,} obs ({df_final['all_naics'].notna().mean()*100:.1f}%) — 2017+ only")
print(f"  Has ANY industry code:   {(df_final['naicsh'].notna() | df_final['sich'].notna() | df_final['all_naics'].notna()).sum():,} obs")

df_final.head(10)


═══ FINAL DATABASE ═══
Shape: (595176, 19)
Unique companies: 46,678
Year range: 1950 – 2025

Industry code coverage:
  naicsh (primary NAICS):  360,001 obs (60.5%)
  sich (primary SIC):      308,070 obs (51.8%)
  all_naics (segment):     49,223 obs (8.3%) — 2017+ only
  Has ANY industry code:   361,801 obs


,gvkey,conm,fyear,datadate,popsrc,naicsh,sich,all_naics,naics_count,naics_sales,at,revt,emp,seq,ceq,mkvalt,sale,ni,csho
0,001000,A & E PLASTIK PAK INC,1961,1961-12-31,D,<NA>,<NA>,NaN,NaN,NaN,<NA>,0.9,<NA>,<NA>,<NA>,<NA>,0.9,<NA>,0.152
1,001000,A & E PLASTIK PAK INC,1962,1962-12-31,D,<NA>,<NA>,NaN,NaN,NaN,<NA>,1.6,<NA>,<NA>,0.552,<NA>,1.6,<NA>,0.181
2,001000,A & E PLASTIK PAK INC,1963,1963-12-31,D,<NA>,<NA>,NaN,NaN,NaN,<NA>,1.457,<NA>,0.553,0.553,<NA>,1.457,0.003,0.186
3,001000,A & E PLASTIK PAK INC,1964,1964-12-31,D,<NA>,<NA>,NaN,NaN,NaN,1.416,2.032,<NA>,0.607,0.607,<NA>,2.032,0.052,0.196
4,001000,A & E PLASTIK PAK INC,1965,1965-12-31,D,<NA>,<NA>,NaN,NaN,NaN,2.31,1.688,<NA>,0.491,0.491,<NA>,1.688,-0.197,0.206
5,001000,A & E PLASTIK PAK INC,1966,1966-12-31,D,<NA>,<NA>,NaN,NaN,NaN,2.43,4.032,<NA>,0.834,0.834,<NA>,4.032,0.164,0.219
6,001000,A & E PLASTIK PAK INC,1967,1967-12-31,D,<NA>,<NA>,NaN,NaN,NaN,2.456,3.594,<NA>,0.744,0.744,<NA>,3.594,-0.09,0.219
7,001000,A & E PLASTIK PAK INC,1968,1968-12-31,D,<NA>,<NA>,NaN,NaN,NaN,5.922,7.4,<NA>,2.571,2.571,<NA>,7.4,0.463,0.372
8,001000,A & E PLASTIK PAK INC,1969,1969-12-31,D,<NA>,<NA>,NaN,NaN,NaN,28.712,37.392,0.9,10.211,10.21,<NA>,37.392,1.766,2.582
9,001000,A & E PLASTIK PAK INC,1970,1970-12-31,D,<NA>,<NA>,NaN,NaN,NaN,33.45,45.335,1.49,10.544,10.544,<NA>,45.335,0.558,2.446


In [28]:

# ── Diagnostic: df_final overview ──
print("═══════════════════════════════════════════")
print("         FINAL DATABASE DIAGNOSTICS")
print("═══════════════════════════════════════════")

print(f"\n📅 Sample period: {df_final['fyear'].min():.0f} – {df_final['fyear'].max():.0f}")
print(f"   Total observations: {len(df_final):,}")
print(f"   Unique companies (gvkey): {df_final['gvkey'].nunique():,}")

# Breakdown by public / private
print(f"\n🏢 By population source:")
for code, label in [('D','Domestic public'), ('O','Other/Private'), ('I','International')]:
    sub = df_final[df_final['popsrc'] == code]
    print(f"   {code} ({label}): {sub['gvkey'].nunique():,} companies, {len(sub):,} obs")

# Industry code coverage
print(f"\n🏷️  Industry code coverage:")
print(f"   naicsh (primary NAICS):   {df_final['naicsh'].notna().sum():,} obs ({df_final['naicsh'].notna().mean()*100:.1f}%)")
print(f"   sich   (primary SIC):     {df_final['sich'].notna().sum():,} obs ({df_final['sich'].notna().mean()*100:.1f}%)")
print(f"   all_naics (seg, 2017+):   {df_final['all_naics'].notna().sum():,} obs ({df_final['all_naics'].notna().mean()*100:.1f}%)")

# Multiple NAICS
has_multi = df_final['naics_count'] > 1
print(f"\n📊 Multiple NAICS (segment-level):")
print(f"   Obs with multiple NAICS:  {has_multi.sum():,} ({has_multi.mean()*100:.1f}%)")
print(f"   Companies with multiple NAICS (ever): "
      f"{df_final[has_multi]['gvkey'].nunique():,}")
print(f"   Distribution of NAICS count (where available):")
print(df_final.loc[df_final['naics_count'].notna() & (df_final['naics_count'] > 0), 'naics_count']
      .value_counts().sort_index().to_string())

# Size variable coverage
print(f"\n📈 Size variable coverage:")
size_vars = ['at', 'revt', 'emp', 'seq', 'ceq', 'mkvalt', 'sale', 'ni', 'csho']
for v in size_vars:
    n = df_final[v].notna().sum()
    pct = df_final[v].notna().mean() * 100
    print(f"   {v:10s}: {n:>10,} obs ({pct:5.1f}%)")

# Observations by decade
print(f"\n📆 Observations by decade:")
df_temp = df_final[df_final['fyear'].notna()].copy()
df_temp['decade'] = (df_temp['fyear'] // 10 * 10).astype(int)
for d, count in df_temp['decade'].value_counts().sort_index().items():
    cos = df_temp[df_temp['decade'] == d]['gvkey'].nunique()
    print(f"   {d}s: {count:>10,} obs, {cos:>6,} companies")


═══════════════════════════════════════════
         FINAL DATABASE DIAGNOSTICS
═══════════════════════════════════════════

📅 Sample period: 1950 – 2025
   Total observations: 595,176
   Unique companies (gvkey): 46,678

🏢 By population source:
   Unique companies (gvkey): 46,678

🏢 By population source:
   D (Domestic public): 46,678 companies, 595,176 obs
   D (Domestic public): 46,678 companies, 595,176 obs
   O (Other/Private): 0 companies, 0 obs
   I (International): 0 companies, 0 obs

🏷️  Industry code coverage:
   naicsh (primary NAICS):   360,001 obs (60.5%)
   sich   (primary SIC):     308,070 obs (51.8%)
   all_naics (seg, 2017+):   49,223 obs (8.3%)

📊 Multiple NAICS (segment-level):
   Obs with multiple NAICS:  23,758 (4.0%)
   Companies with multiple NAICS (ever): 5,089
   Distribution of NAICS count (where available):
naics_count
1.0     14560
2.0     15539
3.0      3613
4.0      2395
5.0      1127
6.0       574
7.0       229
8.0       143
9.0        52
10.0       30
11

In [30]:

# Remove companies that have NO industry code at all (no naicsh, no sich, no segment NAICS)
has_industry = df_final['naicsh'].notna() | df_final['sich'].notna() | df_final['all_naics'].notna()
df_final = df_final[has_industry].copy()

print(f"After removing obs with no industry code:")
print(f"  Shape: {df_final.shape}")
print(f"  Unique companies: {df_final['gvkey'].nunique():,}")
print(f"  Year range: {df_final['fyear'].min():.0f} – {df_final['fyear'].max():.0f}")

df_final.to_csv('/Users/juliaborges/Downloads/compustat_with_naics.csv', index=False)
print(f"\nSaved to compustat_with_naics.csv")


After removing obs with no industry code:
  Shape: (361801, 19)
  Unique companies: 33,462
  Year range: 1950 – 2025

Saved to compustat_with_naics.csv

Saved to compustat_with_naics.csv
